# Notebook 02 — Inspect LongEval-Sci & build the index

WIR 2026 · TH Köln — this notebook mirrors the official tutorial
`pyterrier-irdatasets-indexing` and is the first half of Assignment III
(Stage 3): build the IR system.

LongEval-Sci (CLEF 2026, Task 1) is a scientific ad-hoc retrieval collection built
from the CORE open-access papers. The `snapshot-1/train/dctr` slice we use has
about 869,902 documents and 100 judged queries (click-derived graded relevance).

Goal: Exactly like the tutorial, we `load` the dataset, stream it into an
`IterDictIndexer`, and verify the index has 869,902 documents. We add one thing on
top of the tutorial: we also store the `authors` / `pubyear` metadata in the index
and write a facet document-frequency map (`facet_df.json`). That metadata is the ingredient for our Stage-4 hypothesis (the author power-law boost in notebooks 03/04)
needs — nothing else about the indexing changes.

- The full text (`default_text()` = title + abstract) is indexed for retrieval just as in the tutorial;
- we keep `authors`/`pubyear`/`title` as stored metadata (the abstract is re-fetched from the `ir_datasets` docstore when a re-ranker needs it in notebooks 06/07).

## 0. Configuration

A single place for every path and knob. The other notebooks use the same values.

In [1]:
from pathlib import Path

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

DATA_DIR    = REPO_ROOT / "data"
INDEX_DIR   = REPO_ROOT / "index" / "longeval-sci"     # metadata-rich index lives here
FACET_DF_PATH = INDEX_DIR / "facet_df.json"            # collection DF map for facets

# The judged training/dev slice (the bare "snapshot-1" id has no qrels).
DATASET_ID  = "longeval-sci-2026/snapshot-1/train/dctr"

# Indexing knobs (these ARE part of the experiment — the Text Analysis stage)
STEMMER   = "porter"
STOPWORDS = "terrier"

# fields which the power-law boost reads at re-rank time.
# title: kept for inspection / error analysis (not boosted)
# authors: informetric facet (Lotka author power law) -> expected to HELP
# pubyear: NEGATIVE CONTROL (boosting by year hurts as we learnt)
META_FIELDS  = {"docno": 64, "title": 256, "authors": 512, "pubyear": 8}
FACET_FIELDS = ["authors", "pubyear"]

print("index dir :", INDEX_DIR)
print("dataset   :", DATASET_ID)

index dir : C:\Users\rafae\OneDrive\Desktop\TH_Koeln\03-Semester\Web_Information_Retrieval\WIR_Retriever_Engine\index\longeval-sci
dataset   : longeval-sci-2026/snapshot-1/train/dctr


In [2]:
import pyterrier as pt
from ir_datasets_longeval import load

if not pt.java.started():
    pt.java.init()

dataset = load(DATASET_ID)

Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


## 1. Inspect the documents

`ir_datasets` document objects are namedtuples, so `_fields` lists every attribute.
We look at the field names, one sample document, and the `default_text()` (title +
abstract) that retrievers actually index.

In [3]:
doc = next(dataset.docs_iter())
print("DOCUMENT FIELDS:", doc._fields, "\n")
for f in doc._fields:
    val = str(getattr(doc, f))
    print(f"  {f:<13}: {val[:160]}{'...' if len(val) > 160 else ''}")

print("\n  default_text():", doc.default_text()[:200], "...")

DOCUMENT FIELDS: ('doc_id', 'title', 'abstract', 'authors', 'createdDate', 'doi', 'arxivId', 'pubmedId', 'magId', 'oaiIds', 'links', 'publishedDate', 'updatedDate') 

  doc_id       : 8724
  title        : Two new species of Pterostichus Bonelli subgenus Pseudoferonina Ball (Coleoptera, Carabidae, Pterostichini) from the mountains of central Idaho, U.S.A.
  abstract     : Two new species of Pterostichus Bonelli subgenus Pseudoferonina Ball, are described from the mountains of central Idaho: Pterostichus bousqueti Bergdahl [type l...
  authors      : [{'name': 'Bergdahl,James'}, {'name': 'Kavanaugh,David'}]
  createdDate  : 2014-05-06T17:20:01
  doi          : 10.3897/zookeys.104.1272
  arxivId      : None
  pubmedId     : None
  magId        : None
  oaiIds       : ['10.3897/zookeys.104.1272', 'info:doi/10.3897%2fzookeys.104.1272', 'oai:biodiversitylibrary.org:part/99186', 'oai:doaj.org/article:352a9d063c0b4196b82be81fa43a...
  links        : [{'type': 'display', 'url': 'https://core.a

## 2. Inspect queries and qrels

`topics` are the information needs; `qrels` are the relevance judgments
(`relevance > 0` means relevant). These power the Cranfield-style evaluation in the
next notebooks.

In [5]:
print("first 3 queries")
for i, q in enumerate(dataset.queries_iter()):
    print(f"  {q.query_id}: {q.default_text()}")
    if i >= 2:
        break

print("\nfirst 3 qrels")
for i, qrel in enumerate(dataset.qrels_iter()):
    print(" ", qrel)
    if i >= 2:
        break

n_q  = sum(1 for _ in dataset.queries_iter())   # queries are namedtuples, so len() works
n_qr = sum(1 for _ in dataset.qrels_iter())     # qrels are tuples, so sum() works
print(f"\n{n_q} queries, {n_qr} qrels")

first 3 queries
  d40a44b633a216fc65484a60dadd48ac: utilizing llm to identify entities in natural language text
  1200cc678dce697633576f9941994baa: psychology human behavior
  e54f68f74633d43b86d0247af6197544: mark twain

first 3 qrels
  TrecQrel(query_id='d40a44b633a216fc65484a60dadd48ac', doc_id='141348100', relevance=0, iteration='0')
  TrecQrel(query_id='d40a44b633a216fc65484a60dadd48ac', doc_id='142056361', relevance=0, iteration='0')
  TrecQrel(query_id='d40a44b633a216fc65484a60dadd48ac', doc_id='142128639', relevance=0, iteration='0')

100 queries, 8772 qrels


## 3. Verdict — which facets can we boost?

As we know, the power-law boost only works if the documents expose a power-law–distributed
metadata field. Let's check.

In [6]:
fields = set(doc._fields)
facets = sorted(fields & {"author", "authors", "subject", "subjects", "keywords", "venue"})
dates  = sorted(fields & {"pubyear", "year", "publishedDate", "createdDate", "updatedDate"})
print("informetric facet fields :", facets)
print("date fields (for pubyear) :", dates)

informetric facet fields : ['authors']
date fields (for pubyear) : ['createdDate', 'publishedDate', 'updatedDate']


Verdict: LongEval-Sci 2026 exposes `authors` (a list of `{"name": "Last,First"}`
dicts) but no `subject` / `keyword` field. So:

- `authors` is our positive facet (Lotka's author power law → boosting might
  help), and
- `pubyear` (derived from `publishedDate`) is the negative control — boosting a
  ranking by publication year should *hurt* (we confirm this in notebook 04).

Both go into the index below.

## 4. Helper functions for the metadata facets

`authors` is a list of dicts and each name itself contains a comma like:
(`"Bergdahl,James"`), so we must not split on commas. We join author names with
`"; "` when storing, and derive a 4-digit year from the available date fields.

In [7]:
import re

# let's create some helper functions to extract metadata from the document
def author_names(doc) -> list:
    """Cleaned, lowercased author names for one LongEval-Sci document."""
    names = []
    for a in (getattr(doc, "authors", None) or []):
        name = (a.get("name") or a.get("Name") or "") if isinstance(a, dict) else str(a)
        name = name.strip().lower()
        if name:
            names.append(name)
    return names

def pub_year(doc) -> str:
    """Derive a 4-digit publication year from the available date fields."""
    for attr in ("publishedDate", "createdDate", "updatedDate"):
        val = getattr(doc, attr, None)
        if val:
            m = re.match(r"\s*(\d{4})", str(val))
            if m:
                return m.group(1)
    return ""

def doc_facet_values(doc, field: str) -> list:
    """Facet values for a document field (used to accumulate collection DF)."""
    if field == "authors":
        return author_names(doc)
    if field == "pubyear":
        y = pub_year(doc)
        return [y] if y else []
    raw = getattr(doc, field, None)
    return [p.strip().lower() for p in re.split(r"[;]", str(raw)) if p.strip()] if raw else []

# quick smoke test on the sample doc
print("authors:", author_names(doc)[:5])
print("pubyear:", pub_year(doc))

authors: ['bergdahl,james', 'kavanaugh,david']
pubyear: 2011


## 5. Build the index + facet-DF map

We stream documents into `IterDictIndexer` (exactly the tutorial's step) and, in the
same single pass, accumulate how many documents each author / year appears in (the
facet DF).

Indexing ~870k docs takes rougly 10 minutes. To avoid rebuilding every time,
we designed the cell to reuse an existing index or we set the variable  `FORCE_REBUILD = True` then it starts over again.

- An Interesting remark from GitHub "jueri": If indexing is too heavy on our machine, the organisers publish a prebuilt index: `index = pt.Artifact.from_hf("jueri/longeval-2026-snapshot-1-index")`.
- We build our own so the `authors` / `pubyear` metadata is available for the boost.

In [8]:
import json
from collections import defaultdict

FORCE_REBUILD = False     # set True to rebuild from scratch

def build_index(dataset, index_dir: Path, facet_df_path: Path):
    """Build the Terrier index AND the facet-DF map in one streaming pass."""
    index_dir.mkdir(parents=True, exist_ok=True)
    facet_fields = [f for f in FACET_FIELDS if f in META_FIELDS]
    facet_df = {f: defaultdict(int) for f in facet_fields}
    counter = {"n": 0}

    def docs_iter():
        for d in dataset.docs_iter():
            counter["n"] += 1
            rec = {"docno": d.doc_id, "text": d.default_text() if hasattr(d, "default_text") else getattr(d, "text", "")}
            rec["title"]   = str(getattr(d, "title", "") or "")[:META_FIELDS["title"]]
            rec["authors"] = "; ".join(author_names(d))[:META_FIELDS["authors"]]
            rec["pubyear"] = pub_year(d)[:META_FIELDS["pubyear"]]
            for f in facet_fields:                       # count DF of unique values per doc
                for v in set(doc_facet_values(d, f)):
                    facet_df[f][v] += 1
            yield rec

    indexer = pt.IterDictIndexer(str(index_dir), overwrite=True,
                                 stemmer=STEMMER, stopwords=STOPWORDS, meta=META_FIELDS)
    indexref = indexer.index(docs_iter())

    out = {"_n_docs": counter["n"]}
    for f in facet_fields:
        out[f] = dict(facet_df[f])
    facet_df_path.write_text(json.dumps(out))
    return indexref

if FORCE_REBUILD or not (INDEX_DIR / "data.properties").exists():
    print("Building index ...")
    build_index(dataset, INDEX_DIR, FACET_DF_PATH)
    print("done.")
else:
    print(f"Index already present at {INDEX_DIR} — reusing it.")

Index already present at C:\Users\rafae\OneDrive\Desktop\TH_Koeln\03-Semester\Web_Information_Retrieval\WIR_Retriever_Engine\index\longeval-sci — reusing it.


## 6. Let's verify the build

We expect about 869,902 documents and a vocabulary of 1.5M terms.

In [9]:
index = pt.IndexFactory.of(str(INDEX_DIR))
print(index.getCollectionStatistics().toString())
print("stored metadata keys:", list(index.getMetaIndex().getKeys()))

Number of documents: 869902
Number of terms: 1529555
Number of postings: 83855040
Number of fields: 0
Number of tokens: 140144709
Field names: []
Positions:   false

stored metadata keys: ['docno', 'title', 'authors', 'pubyear']


## 7. The facet-DF map (the IDF ingredient :))

`facet_df.json` maps every facet value to the number of documents it appears in. In
notebook 03 we turn this into an IDF — a rare author is more informative than a
prolific one (we saw that in: "a rare shared citation matters more than a common one").
The author distribution below is the classic informetric power law: a few authors
appear in many papers, most appear in just one.

In [10]:
facet = json.loads(FACET_DF_PATH.read_text())
n_docs = facet.pop("_n_docs")
print(f"N documents: {n_docs}")
for f in FACET_FIELDS:
    print(f"  {f:<8}: {len(facet.get(f, {})):>8,} distinct values")

# the most prolific authors (head of the power law)
top_authors = sorted(facet["authors"].items(), key=lambda x: x[1], reverse=True)[:10]
import pandas as pd
display(pd.DataFrame(top_authors, columns=["author", "doc_frequency"]))

# share of authors that appear in exactly one document (the long tail)
singletons = sum(1 for v in facet["authors"].values() if v == 1)
print(f"\nauthors in exactly 1 doc: {singletons:,} "
      f"({100*singletons/len(facet['authors']):.1f}% of all authors)")

N documents: 869902
  authors : 2,696,925 distinct values
  pubyear :      600 distinct values


,author,doc_frequency
0,cincinnati (ohio). health dept.,1127
1,canal society of ohio,1104
2,wang,769
3,lee,657
4,chen,639
5,zhang,619
6,li,603
7,"lópez silván, javier",601
8,"aparicio martín, carlos",600
9,"rodrigo torices, julia",600



authors in exactly 1 doc: 2,238,480 (83.0% of all authors)


> **Remarks:** We have a verified 869,902-document index that also
> stores `authors`/`pubyear`, plus a `facet_df.json` confirming the author power law.